In [ ]:
import os, glob, gc, csv, yaml
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.manifold import TSNE

import torch
from torchvision.transforms import Compose, Resize, CenterCrop

from utils.processing import make_normalize
from networks import create_architecture, load_weights

# UMAP optional
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

print("Imports done. UMAP available:", UMAP_AVAILABLE)

In [ ]:
# ====== 환경설정 ======
BASE_DIR = "/home/ice06/project/HS/Dataset_0223/Case1" #Case2
WEIGHTS_DIR = "./weights"
MODEL_NAME = "Corvi"
OUTPUT_PREFIX = "/home/project/t-SNE_Corvi+_result"

DATASET_CONFIG = {
    "Real_TN": ["TN"],
    "Real_FP": ["FP"],
    "Fake_TP": ["TP"],
    "Fake_FN": ["FN"],
}

IMAGES_PER_GROUP = 10000
USE_CPU = False #GPU 안되는 경우 CPU 
FEATURE_TYPE = "fc_input"

In [ ]:
from pathlib import Path

def assert_exists(p, name):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f" {name} not found: {p}")
    print(f" {name}: {p}")
    return p

assert_exists(BASE_DIR, "BASE_DIR")

# dataset folders check
for k, folders in DATASET_CONFIG.items():
    for folder in folders:
        assert_exists(Path(BASE_DIR) / folder, f"Dataset folder ({k})")

# weights config check
assert_exists(Path(WEIGHTS_DIR) / MODEL_NAME / "config.yaml", "Model config.yaml")
print(" All path checks passed")

In [ ]:
def get_config(model_name, weights_dir='./weights'):
    with open(os.path.join(weights_dir, model_name, 'config.yaml')) as fid:
        data = yaml.load(fid, Loader=yaml.FullLoader)
    model_path = os.path.join(weights_dir, model_name, data['weights_file'])
    return data['model_name'], model_path, data['arch'], data['norm_type'], data['patch_size']


def build_transform(patch_size, norm_type):
    tfms = []
    if patch_size == 'Clip224':
        tfms.extend([Resize(224), CenterCrop((224, 224))])
    elif isinstance(patch_size, (tuple, list)):
        tfms.extend([Resize(*patch_size), CenterCrop(patch_size[0])])
    elif patch_size is not None and patch_size > 0:
        tfms.append(CenterCrop(patch_size))
    tfms.append(make_normalize(norm_type))
    return Compose(tfms)


def clear_gpu_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()


def extract_features_for_tsne(
    base_dir,
    weights_dir,
    model_name,
    output_prefix,
    dataset_config,
    images_per_group=100,
    use_cpu=False,
    feature_type="fc_input",
):
    if use_cpu:
        device = torch.device('cpu')
        print("Running on CPU")
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    clear_gpu_memory()
    
    _, model_path, arch, norm_type, patch_size = get_config(model_name, weights_dir)
    model = load_weights(create_architecture(arch), model_path)
    model = model.to(device).eval()
    transform = build_transform(patch_size, norm_type)
    
    cache = {}
    
    def conv_hook(module, input, output):
        cache["conv_output"] = output.detach().cpu()
    
    def fc_hook(module, input, output):
        cache["fc_input"] = input[0].detach().cpu()
    
    if hasattr(model, 'layer4'):
        target_layer = model.layer4[-1]
    elif hasattr(model, 'features'):
        target_layer = model.features[-1]
    else:
        raise ValueError("Cannot find target conv layer")
    
    conv_handle = target_layer.register_forward_hook(conv_hook)
    fc_handle = model.fc.register_forward_hook(fc_hook)
    
    output_dir = f"{output_prefix}_tsne_analysis"
    os.makedirs(output_dir, exist_ok=True)
    
    print("\nExtracting features...")
    
    all_features, all_labels, all_scores, all_paths = [], [], [], []
    groups = list(dataset_config.keys())
    
    for group in groups:
        folders = dataset_config[group]
        all_img_paths = []
        
        for folder in folders:
            folder_path = os.path.join(base_dir, folder)
            if os.path.exists(folder_path):
                imgs = glob.glob(os.path.join(folder_path, "*.*"))
                imgs = [p for p in imgs if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
                all_img_paths.extend(imgs)
            else:
                print(f"  Folder not found: {folder_path}")
        
        all_img_paths = all_img_paths[:images_per_group]
        print(f"  {group}: {len(all_img_paths)} images")
        
        for img_path in tqdm(all_img_paths, desc=f"Processing {group}"):
            try:
                img_pil = Image.open(img_path).convert("RGB")
                img_tensor = transform(img_pil).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    output = model(img_tensor)
                    if output.shape[-1] == 1 or output.ndim == 1 or output.shape[1] == 1:
                        score = torch.sigmoid(output).squeeze().item()
                    else:
                        score = torch.softmax(output, dim=1)[0, 1].item()
                
                if feature_type == "fc_input":
                    feature = cache["fc_input"].numpy().squeeze()
                else:
                    feature = cache["conv_output"].numpy().squeeze().reshape(-1)
                
                all_features.append(feature)
                all_labels.append(group)
                all_scores.append(score)
                all_paths.append(img_path)
                
            except Exception as e:
                print(f"Error: {img_path} - {e}")
                continue
        
        clear_gpu_memory()
    
    conv_handle.remove()
    fc_handle.remove()
    
    features = np.array(all_features)
    labels = np.array(all_labels)
    scores = np.array(all_scores)
    
    print(f"\nTotal features extracted: {features.shape}")
    
    if len(features) == 0:
        print(" 추출된 특징이 없습니다. 경로 설정을 다시 확인해주세요.")
        return None

    print("\nRunning T-SNE...")
    
    tsne = TSNE(
        n_components=2,
        perplexity=min(30, len(features) - 1),
        learning_rate='auto',
        init='pca',
        random_state=42,
        n_iter=1000
    )
    features_tsne = tsne.fit_transform(features)
    
    color_map = {
        "Real_TN": "#2ecc71",
        "Real_FP": "#f39c12",
        "Fake_TP": "#3498db",
        "Fake_FN": "#e74c3c",
    }
    marker_map = {
        "Real_TN": "o",
        "Real_FP": "D",
        "Fake_TP": "^",
        "Fake_FN": "s",
    }
    
    plt.figure(figsize=(12, 10))
    for group in groups:
        mask = labels == group
        plt.scatter(
            features_tsne[mask, 0],
            features_tsne[mask, 1],
            c=color_map.get(group, "#95a5a6"),
            marker=marker_map.get(group, "o"),
            label=f"{group} (n={mask.sum()})",
            alpha=0.7,
            s=50,
            edgecolors='white',
            linewidths=0.5
        )
    
    plt.xlabel("T-SNE Dimension 1", fontsize=12)
    plt.ylabel("T-SNE Dimension 2", fontsize=12)
    plt.title("T-SNE Visualization: 4 Classes Comparison", fontsize=14)
    plt.legend(loc='best', fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{output_dir}/tsne_visualization.png", dpi=200, bbox_inches='tight')
    plt.close()
    
    print(f" Saved: {output_dir}/tsne_visualization.png")
    
    np.savez(
        f"{output_dir}/tsne_data.npz",
        features=features,
        features_tsne=features_tsne,
        labels=labels,
        scores=scores
    )

    print(f"\n Core outputs saved to: {output_dir}/")
    return {
        "features": features,
        "features_tsne": features_tsne,
        "labels": labels,
        "scores": scores,
        "paths": all_paths,
        "output_dir": output_dir, 
    }

In [ ]:
#-----------------정탐에 포함된 하드 샘플 추가 확인 -----------------
def export_tsne_and_extract_mixed_fn(
    results,
    y_threshold=-10,
    fn_label="Fake_FN",
    csv_name="tsne_coordinates_full.csv",
    txt_name="mixed_hard_samples.txt"
):
    features_tsne = results["features_tsne"]
    labels = results["labels"]
    scores = results["scores"]
    paths = results["paths"]
    output_dir = results["output_dir"]

    csv_path = os.path.join(output_dir, csv_name)
    txt_path = os.path.join(output_dir, txt_name)

    mixed_fn_count = 0

    print("\n🔍 Extracting Hard Samples in the Blue Region...")
    print(f"   기준: {fn_label} and T-SNE Y > {y_threshold}")

    with open(csv_path, 'w', newline='', encoding='utf-8') as f_csv, \
         open(txt_path, 'w', encoding='utf-8') as f_txt:

        writer = csv.writer(f_csv)
        writer.writerow(["File_Path", "Group", "Score", "TSNE_X", "TSNE_Y"])

        f_txt.write(f"=== Fake_TP 군집에 섞여 있는 {fn_label} 이미지 목록 ===\n")
        f_txt.write(f"기준: T-SNE Y좌표 > {y_threshold}\n\n")

        for i in range(len(labels)):
            x, y = features_tsne[i, 0], features_tsne[i, 1]
            writer.writerow([paths[i], labels[i], scores[i], x, y])

            if labels[i] == fn_label and y > y_threshold:
                f_txt.write(f"경로: {paths[i]} | X: {x:.2f}, Y: {y:.2f}\n")
                mixed_fn_count += 1

    print(f" Saved Full CSV: {csv_path}")
    print(f" Found {mixed_fn_count} {fn_label} images mixed by criterion!")
    print(f" Saved Mixed List: {txt_path}")

    return {
        "csv_path": csv_path,
        "txt_path": txt_path,
        "mixed_fn_count": mixed_fn_count
    }

In [ ]:
results = extract_features_for_tsne(
    base_dir=BASE_DIR,
    weights_dir=WEIGHTS_DIR,
    model_name=MODEL_NAME,
    output_prefix=OUTPUT_PREFIX,
    dataset_config=DATASET_CONFIG,
    images_per_group=IMAGES_PER_GROUP,
    use_cpu=USE_CPU,
    feature_type=FEATURE_TYPE,
)

In [ ]:
if results is None:
    print(" results가 None이라 추가 분석을 스킵")
else:
    aux = export_tsne_and_extract_mixed_fn(results, y_threshold=-10, fn_label="Fake_FN")
    aux